# Trabalho Final ALC 2026.1 — Compressão de tokens via SVD

Notebook de **análise compilada**: percorre o pipeline do trabalho ponta a ponta e valida os resultados do artigo.

**Tempo estimado (CPU):** ~20 s embeddings + ~2 min grade = **~3 min** na primeira execução.

## Visão geral do pipeline

1. **Carrego os dados** — IMDb balanceado, split estratificado 70/30
2. **Gero $F$ por review** — tokenizo → encoder MiniLM → hidden states → filtro → matriz $F \in \mathbb{R}^{T \times D}$
3. **Podo $F$** — 4 operadores, orçamento $\rho$
4. **Avalio** — pooling + Ridge → acurácia downstream; métricas algébricas ($E_k$, $R_k$, $C$)
5. **Agrego a grade** — 4 métodos × 4 $\rho$ = 16 runs → tabelas e figura

Detalhes da extração de $F$ (passo a passo de [`embeddings.py`](../src/embeddings.py)): ver [`extracao_embedding.ipynb`](extracao_embedding.ipynb).
Detalhes dos operadores de poda (passo a passo de [`compressores.py`](../src/compressores.py)): ver [`compressao_tokens.ipynb`](compressao_tokens.ipynb).
Detalhes das métricas (passo a passo de [`metricas.py`](../src/metricas.py)): ver [`avaliacao_metricas.ipynb`](avaliacao_metricas.ipynb).

## Perguntas de pesquisa (metodologia)

1. Leverage score preserva acurácia competitiva frente a heurísticas locais?
2. `svd_energia` difere de `svd`?
3. $E_k^{\mathrm{energia}}$ permanece estável quando $\rho$ varia?

## Resultados-alvo (seção Resultados do artigo)

- Tabela de acurácia downstream por método e $\rho$
- $E_k^{\mathrm{energia}} \approx 0{,}951$ (SVD, invariante a $\rho$)
- Figura trade-off acurácia × compressão

**Dependências:** `numpy`, `matplotlib` (requirements.txt) + `datasets`, `transformers`, `torch` para IMDb.


In [ ]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

_raiz = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "notebook_setup.py").exists()
)
sys.path.insert(0, str(_raiz))
import notebook_setup
notebook_setup.configurar()

import dados
import embeddings
import compressores as cp
import metricas
import experimento
import visualizacao
from compressores import COMPRESSORES
from constantes import MAX_AMOSTRAS, MAX_TOKENS, N_POR_CLASSE, ORCAMENTOS, PASTA_RESULTADOS, SEED
from experimento import agregar, rodar_grade, salvar_csv

print("Diretório de trabalho:", os.getcwd())
print("Saída CSV:", os.path.join(PASTA_RESULTADOS, "resultados.csv"))

## 1. `dados.py` — Corpus IMDb

**Metodologia (§ Corpus):** subconjunto balanceado de **400 reviews** (200 negativas, 200 positivas), split estratificado 70/30. A unidade experimental é cada review → uma matriz $F^{(i)} \in \mathbb{R}^{T_i \times D}$.

**Por que este módulo?** Isola o carregamento do corpus e o split reprodutível, sem misturar lógica de embedding ou compressão.

Nesta seção também **materializamos todos os embeddings** uma única vez; as seções seguintes reutilizam `amostras`.


In [ ]:
textos, rotulos = dados.carregar_imdb(n_por_classe=N_POR_CLASSE, max_amostras=MAX_AMOSTRAS)
idx_treino, idx_teste = dados.dividir_indices_estratificado(rotulos, semente=SEED)

n_neg = int(np.sum(rotulos == 0))
n_pos = int(np.sum(rotulos == 1))
print(f"Total: {len(textos)} reviews ({n_neg} neg, {n_pos} pos)")
print(f"Treino: {len(idx_treino)} | Teste: {len(idx_teste)}")
print(f"Treino por classe: neg={np.sum(rotulos[idx_treino]==0)}, pos={np.sum(rotulos[idx_treino]==1)}")
print(f"Teste por classe:  neg={np.sum(rotulos[idx_teste]==0)}, pos={np.sum(rotulos[idx_teste]==1)}")

print(f"\nGerando embeddings para as {len(textos)} reviews (max_tokens={MAX_TOKENS})...")
t0_embed = time.perf_counter()
amostras = [
    embeddings.texto_para_embedding(texto, max_tokens=MAX_TOKENS)
    for texto in textos
]
print(f"Embeddings prontos em {time.perf_counter() - t0_embed:.0f} s")

## 2. `embeddings.py` — Matriz $F$ (resumo)

**Metodologia (§ Construção da matriz):** cada review vira $F \in \mathbb{R}^{T \times D}$ via `all-MiniLM-L6-v2` (`last_hidden_state`, token-level), remoção de [CLS]/[SEP]/padding, `max_tokens=256`, $D=384$.

Pipeline em uma linha: **texto → tokenização → encoder → hidden states → filtro → $F$**.

**Por que token-level?** O trabalho poda **linhas** de $F$; pooling de sentença eliminaria essa dimensão.

Usamos `amostras` já calculadas na seção 1 (sem re-embedding). Para o passo a passo: [`extracao_embedding.ipynb`](extracao_embedding.ipynb).


In [ ]:
F_exemplo = amostras[0]
print("Review 0 — shape F (T x D):", F_exemplo.shape)
print("D (dimensão embedding):", F_exemplo.shape[1], "(esperado 384)")

ts_todos = [F.shape[0] for F in amostras]
print(
    f"T médio ({len(amostras)} reviews): {np.mean(ts_todos):.1f}  "
    f"(min={min(ts_todos)}, max={max(ts_todos)})"
)

## 3. `compressores.py` — Operadores de poda

**Metodologia (§ Baselines e SVD):** dado orçamento $\rho$, mantemos $T' = \max(1, \lfloor \rho T \rfloor)$ linhas. Quatro operadores sob o **mesmo** $\rho$:

| Método | Critério |
|--------|----------|
| `full` | sem poda (teto) |
| `random` | $T'$ tokens aleatórios |
| `svd` | leverage $\ell_i/k$ |
| `svd_energia` | leverage ponderado por $\Sigma_k$ |

**Por que este módulo?** Centraliza os operadores algébricos; treino e teste passam pelo mesmo operador e $\rho$.

Demonstração abaixo usa $F$ da review 0 (§2). 

Obs.: K depende do percentual de energia definida que vamos usar, definido em `variancia_explicada` ; Por isso nÃo depende de rho ou T' ; É calculado com a matriz inteira


In [ ]:
F_demo = F_exemplo
rho_demo = 0.5
print(f"Demonstração em review real (T={F_demo.shape[0]}, D={F_demo.shape[1]}, rho={rho_demo}):\n")

for nome, compressor in COMPRESSORES.items():
    indices, F_podado, info = compressor(F_demo, rho_demo, semente=0)
    C = metricas.compressao(F_demo.shape[0], len(indices))
    extra = ""
    if nome in ("svd", "svd_energia"):
        extra = f" | k={info['k']} | E_k={metricas.energia_espectral_preservada(info):.3f}"
    print(f"  {nome:12s} -> T'={len(indices):3d} | C={C:.3f}{extra}")

## 4. `metricas.py` — Avaliação dual

**Metodologia (§ Protocolo):** separamos duas avaliações:

- **(A) Métricas algébricas:** compressão $C = 1 - T'/T$; para SVD, $E_k^{\mathrm{energia}}$ sobre $F_k$ e fidelidade $R_k$.
- **(B) Downstream:** pooling médio + normalização $L_2$ → classificador Ridge ($\lambda=1$) → acurácia.

**Ponto central:** $E_k^{\mathrm{energia}}$ mede o subespaço truncado $F_k$, **não** a qualidade da seleção de índices $S$.

Abaixo: uma execução `svd` com $\rho=0{,}5$ sobre **todo o corpus** (mesmo protocolo da grade).


In [ ]:
from experimento import _processar_amostras

rho_demo_metricas = 0.5
run_svd = _processar_amostras(
    amostras, rotulos, "svd", rho_demo_metricas,
    idx_treino, idx_teste, semente=0,
)
_, _, info_svd = cp.comprimir_svd(amostras[0], rho_demo_metricas)

print(f"Experimento completo (n={len(amostras)}, svd, rho={rho_demo_metricas}):")
print(f"  Acurácia downstream: {run_svd['acuracia_downstream']:.3f}")
print(f"  Compressão C:          {run_svd['compressao']:.3f}")
print(f"  E_k^energia (média):   {run_svd['energia_espectral']:.3f}")
print(f"  R_k (média):           {run_svd['fidelidade_reconstrucao']:.3f}")
print(f"  E_k^energia (review 0): {metricas.energia_espectral_preservada(info_svd):.3f}")
print(f"  R_k (review 0):         {metricas.fidelidade_reconstrucao(amostras[0], info_svd['reconstrucao']):.3f}")

## 5. `experimento.py` — Grade completa

**Metodologia (§ Protocolo experimental):** grade **4 métodos × 4 $\rho$ = 16 execuções**. Reutilizamos os embeddings já calculados na seção 1.

Os CSVs são gravados em `resultados/` ao final da execução.


In [ ]:
CSV_SAIDA = os.path.join(PASTA_RESULTADOS, "resultados.csv")

print(
    f"Executando grade completa ({len(amostras)} reviews, "
    f"{len(COMPRESSORES)} métodos, {len(ORCAMENTOS)} rho)..."
)
t0 = time.perf_counter()
linhas = rodar_grade(
    max_amostras=MAX_AMOSTRAS,
    max_tokens=MAX_TOKENS,
    amostras=amostras,
    rotulos=rotulos,
)
os.makedirs(PASTA_RESULTADOS, exist_ok=True)
salvar_csv(linhas, CSV_SAIDA)
salvar_csv(linhas, os.path.join(PASTA_RESULTADOS, "resultados_imdb.csv"))
print(f"Grade concluída em {time.perf_counter() - t0:.0f} s")
print(f"CSV salvo: {CSV_SAIDA} ({len(linhas)} linhas)")

agregado = agregar(linhas)
print(f"\nMétodos agregados: {sorted(agregado.keys())}")

## 6. Resultados — Tabelas, figura e validação

**Metodologia (§ Resultados):** comparamos acurácia downstream por método e $\rho$; reportamos $E_k^{\mathrm{energia}}$ para SVD; figura trade-off acurácia × compressão.


In [ ]:
def formatar_acuracia(ponto, metodo):
    return f"{ponto['acuracia_media']:.3f}"


def tabela_acuracia(agregado):
    metodos_ordem = ["full", "random", "svd", "svd_energia"]
    cabecalho = f"{'Método':<12}" + "".join(f"rho={r:<6.3f}" for r in ORCAMENTOS)
    print("Tabela — Acurácia downstream (IMDb)")
    print(cabecalho)
    print("-" * len(cabecalho))
    for metodo in metodos_ordem:
        if metodo not in agregado:
            continue
        por_rho = {p["orcamento"]: p for p in agregado[metodo]}
        vals = [formatar_acuracia(por_rho[r], metodo) for r in ORCAMENTOS]
        print(f"{metodo:<12}" + "".join(f"{v:<12}" for v in vals))


tabela_acuracia(agregado)


In [ ]:
ek_vals = [
    p["energia_espectral_media"]
    for p in agregado["svd"]
    if not np.isnan(p["energia_espectral_media"])
]
ek_media = float(np.mean(ek_vals))
print(f"E_k^energia (svd, média sobre rho): {ek_media:.3f}  (artigo: 0.951)")

ponto_svd_0125 = next(p for p in agregado["svd"] if p["orcamento"] == 0.125)
print(f"T̄ (tokens/review):     {ponto_svd_0125['t_medio']:.1f}  (artigo: ~209)")
print(f"T̄' @ rho=0.125 (svd):  {ponto_svd_0125['t_linha_medio']:.1f}  (artigo: ~26)")

In [ ]:
caminho_fig = os.path.join(PASTA_RESULTADOS, "tradeoff_acerto_compressao.png")
visualizacao.salvar_grafico_tradeoff(agregado, caminho_fig)

plt.figure(figsize=(8, 5))
for metodo, pontos in sorted(agregado.items()):
    pts = sorted(pontos, key=lambda p: p["compressao"])
    x = [p["compressao"] for p in pts]
    y = [p["acuracia_media"] for p in pts]
    plt.plot(
        x, y, marker="o",
        label=visualizacao.ROTULOS.get(metodo, metodo),
        color=visualizacao.CORES.get(metodo, "#333333"),
    )
plt.xlabel("Compressão (1 - T'/T)")
plt.ylabel("Acurácia downstream")
plt.title("Trade-off Acurácia × Compressão (baselines + SVD)")
plt.legend(fontsize=8, loc="best")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f"Figura salva: {caminho_fig}")